# *<center> Time-lag (delayed-extraction) autotune </center>*

**Purpose.** Derive and demonstrate Wiley–McLaren time-lag focusing on the
refined oa-TOF example — *outside* the GUI (co-tuning is a
demonstration, not a UI feature). The GUI exposes the knobs (pulse delay τ on
the Voltages tab, source temperature on the Ion Source tab); **this notebook
shows why the knobs must move together** and produces the tuned pair.

**The physics in three sentences.** Turn-around time Δt = 2√(mkT)/(qE₁) is
the one packet-width term neither space nor energy focusing can touch, because
two ions at ±v_y are identical in everything but arrival. Delayed extraction
fills field-free for τ, so ±v_y becomes ±v_y·τ of *position* — which fields
**can** focus. The catch: at the matched design the detector is already the
space-focus image (dT/dy₀ = 0), so converted position buys nothing until the
**mirror is detuned** to put dT/dU ≠ 0 back — τ and the mirror scale are one
tuning pair, never two knobs.

**Certified anchors** (population σ, seed 1, 0.5 mm cells): static 77 K
example R 4195/3908 (m/z 100/500); time-lag variant τ=2.40 µs, mirror ×1.040,
0.1 mm sheet → **R 8473** at m/z 500 (0.83 ns), 2769 at m/z 100 (designed
off-mass trade).

## The instrument, before any statistics

The device this notebook flies, drawn from the **solver's own electrode mask** (not a redrawing) with example ion paths exactly as flown. You are looking at the pusher and two-grid source, the field-free tube, the ring-ladder mirror, and the detector beside the source, with example ions tracing the full chevron.

Deck: `examples/reflectron_tof_oa_refined_wm_matched.json`. A geometry figure is not decoration — if the picture and the solved model can disagree, every number below is unverifiable.

In [ ]:
# ---- import-origin guard (run me FIRST) --------------------------------
# THIS notebook belongs to a repo; it must run against THAT repo's
# ion_gym, not whatever `import ion_gym` happens to find (a pip-installed
# copy, or an old tree on PYTHONPATH). A version mismatch does not fail
# politely -- it surfaces mid-run as a confusing AttributeError on some
# API the stale copy predates. So: locate the repo from this notebook's
# working directory, put it FIRST on sys.path, then verify the imported
# package actually came from here -- and REFUSE with the remedy if not.
# GENERATED CELL: every notebook carries one identical copy, stamped from
# a single definition in the development tree. Edits made here are
# overwritten the next time the notebook is regenerated.
import sys
from pathlib import Path

# leading underscores here are DELIBERATE (charter: stated reason): this
# cell is stamped into every notebook and must not collide with or
# pollute the study's own names
_here = Path.cwd().resolve()
ROOT_GUARD = next((p for p in (_here, *_here.parents)
                   if (p / "ion_gym").is_dir() and (p / "notebooks").is_dir()),
                  None)
if ROOT_GUARD is None:
    raise RuntimeError(
        f"cannot locate the ion_gym repo at or above {_here}; start the "
        f"kernel in the repo root or in a folder inside it")
if str(ROOT_GUARD) not in sys.path:
    sys.path.insert(0, str(ROOT_GUARD))

import ion_gym
_origin = Path(ion_gym.__file__).resolve().parent
if _origin.parent != ROOT_GUARD:
    raise RuntimeError(
        f"imported ion_gym v{ion_gym.__version__} from {_origin}, which is "
        f"NOT this repo ({ROOT_GUARD / 'ion_gym'}). Either the kernel "
        f"already imported a stale copy (restart the kernel and run this "
        f"cell first) or another copy shadows the repo (a pip-installed "
        f"ion_gym: `pip uninstall ion_gym`; or a stale PYTHONPATH entry). "
        f"Refusing now beats an AttributeError several cells later.")
print(f"ion_gym v{ion_gym.__version__} · loaded from {_origin}")


<!-- origin-guard-note -->
The cell above only pins this notebook to its own repo's `ion_gym`. The notebook proper begins below.


In [ ]:
from ion_gym.viz.nb_panels import show_instrument_panel
# banked panel when present, else rendered live from the deck
# of record (self-contained on a public checkout).
show_instrument_panel('examples/reflectron_tof_oa_refined_wm_matched.json', banked='panel_tof.png', height=760)


In [ ]:
from pathlib import Path
import json, math
import numpy as np

# repo-relative, never absolute (works from notebooks/ or repo root)
ROOT = Path.cwd() if (Path.cwd()/"examples").is_dir() else Path.cwd().parent
SPEC = ROOT/"examples"/"reflectron_tof_oa_refined_wm_matched.json"
spec_d = json.loads(SPEC.read_text())
els = {e["name"]: e for e in spec_d["geometry"]["electrodes"]}

E_CHG, KG_AMU, KB = 1.602176634e-19, 1.66053907e-27, 1.380649e-23
# geometry/fields read OFF THE SPEC, not retyped
yG1 = els["extract_G1"]["shapes"][0]["y_mm"]
yG2 = els["extract_G2"]["shapes"][0]["y_mm"]
y_ent = els["mirror_entrance"]["shapes"][0]["y_mm"]
det = els["detector"]["shapes"][0]
y_det = det["y_mm"] + det["height_mm"]
V_G1 = els["extract_G1"]["dc"]; V_push = els["pusher"]["dc"]
y_push_top = els["pusher"]["shapes"][0]["y_mm"] + els["pusher"]["shapes"][0]["height_mm"]
E1 = (V_push - V_G1)/(yG1 - y_push_top)
E2 = V_G1/(yG2 - yG1)
# mirror gradient from the ladder itself (rule V(y)=E_mir*(y-y_ent))
r1 = els["mir_ring01"]; EM0 = r1["dc"]/(r1["shapes"][0]["y_mm"]
                                        + r1["shapes"][0]["height_mm"]/2 - y_ent)
print(f"E1 {E1:.1f} V/mm, E2 {E2:.2f} V/mm, E_mir {EM0:.3f} V/mm  (from spec)")

# ---- deck-inherited physics: visible and overridable -----------------
# The deck loaded below supplies the drive, the ion, the gas and the
# integration settings. Leave an entry None to INHERIT it from the deck;
# set one to OVERRIDE. Whatever ends up in force is printed, so this
# notebook's output always states its own operating point.
# (One namespaced dict, not loose globals: the first version used bare
# names like KE_EV and N_IONS, which collided with the parameters these
# notebooks already own -- and silently changed them.)
DECK_OVERRIDES = dict(
    rf_v=None, rf_f=None,        # confining-drive amplitude (V) / freq (Hz)
    mz_list=None, charge=None,   # e.g. [622.0] / 1
    ke_ev=None,                  # (lo, hi) eV
    source_t_k=None,             # K, thermal spread of initial velocities
    n_ions=None,                 # ions flown
    gas_on=None, gas=None,       # True/False / e.g. "N2"
    p_torr=None, gas_t_k=None,   # buffer-gas pressure (Torr) / temp (K)
    dt_ns=None, t_max_us=None,   # integration step (ns) / flight time (us)
)


## The ladder: toy → optimized, every rung flown

Before the mechanism, the progression. Each stage changes **one thing**, is
flown in the real solver (24 ions/mass, seed 1, 0.5 mm cells, population σ,
TOF from the pulse where one exists), and names the term that limits it —
which is what tells you the *next* rung. The numbers are produced by this
cell, not quoted.

In [ ]:
# Repo-relative root: every path in this cell derives from it, so the
# notebook runs on any machine and in any cell order.
from pathlib import Path as _P
from ion_gym.io.paths import repo_root as _rr
ROOT = _P(_rr())
import copy
from ion_gym.io.sim_spec import SimSpec
from ion_gym.physics.sim_build import build_run
from ion_gym.physics import ensemble_driver as ed

def fly_stage(path, mutate=None, tau=0.0):
    d = json.loads((ROOT/"examples"/path).read_text())
    if mutate: mutate(d)
    sp = SimSpec.from_dict(d); assert not sp.validate()
    model, fly_fn, cols, births = build_run(sp)
    prog = ed.run(births.shape[0], fly_fn, keep_full=False)
    out = {}
    for mz in (100.0, 500.0):
        tt = np.array([r.summary["tof"]-tau for r in prog.results
                       if r.summary["mz"] == mz and r.summary["y_end"] > 29.5])
        fw = 2.3548*tt.std()
        out[mz] = (len(tt), fw*1e3, tt.mean()/(2*fw))
    return out

def hot(d):    d["source"]["temperature_k"] = 300.0
def sheet(d):  d["source"]["box_mm"] = [3.0, 0.1, 0.0]
def lag(d):
    # the shipped tuned pair (clearance-capped): tau 2.40 us, mirror x1.0400
    step = dict(frequency_hz=1.0, phase_deg=0.0, waveform="table",
                table_t_us=[0.0, 2.40], table_v=[0.0, 1.0],
                interp="hold", pe_mode="instant")
    d["geometry"]["rf_groups"] = [
        dict(name="PUSH", amplitude_v=5500.0, **step),
        dict(name="EXTR", amplitude_v=2500.0, **step)]
    for e in d["geometry"]["electrodes"]:
        if e["name"] == "pusher":     e["rf_groups"] = ["PUSH"]; e["dc"] = 0.0
        if e["name"] == "extract_G1": e["rf_groups"] = ["EXTR"]; e["dc"] = 0.0
        if e["name"].startswith("mir_ring") or e["name"] == "mirror_back":
            e["dc"] = round(e["dc"]*1.0400, 3)
        if e["name"] == "detector":
            e["shapes"][0]["x_mm"] = 19.6; e["shapes"][0]["width_mm"] = 12.6  # trimmed frame (x' = x - 55)
    d["source"]["box_mm"] = [3.0, 0.1, 0.0]
    d["integration"]["dt_ns"] = 1.0

REF = "reflectron_tof_oa_refined_wm_matched.json"
stages = [
 ("A toy (pinned-ring mirror)", "reflectron_tof_oa_with_detector.json",
  None, 0.0, "grid-pinned staircase mirror + weak push (15 V/mm)"),
 ("B refined, 300 K", REF, hot, 0.0,
  "turn-around 1.31/2.93 ns (2*sqrt(mkT)/qE1)"),
 ("C cooled 77 K", REF, None, 0.0,
  "turn-around 0.66/1.48 ns + 2nd-order energy emerging"),
 ("D + 0.1 mm sheet", REF, lambda d: (sheet(d)), 0.0,
  "turn-around (sheet shrinks 2nd-order, not turn-around)"),
 ("E + time-lag (2.40 us, x1.0400)", REF, lag, 2.40,
  "solver-field residuals; off-mass trade is designed"),
]
print(f"{'stage':34s} {'m/z100: FWHM/R':>18s} {'m/z500: FWHM/R':>18s}  limited by")
for name, path, mut, tau, lim in stages:
    r = fly_stage(path, mut, tau)
    print(f"{name:34s} {r[100.][1]:6.2f} ns {r[100.][2]:6.0f}"
          f"  {r[500.][1]:8.2f} ns {r[500.][2]:6.0f}   {lim}")
    assert r[100.][0] == 24 and r[500.][0] == 24, "transmission lost"
print()
print("Stage A's numbers carry its OWN caveat: the toy's resolution is not")
print("converged under cell refinement (its notes: R 372->65 at 0.25 mm);")
print("its row is the honest 0.5 mm value of a model with a known field")
print("artifact. Every later stage IS refinement-converged (<=0.04 ns).")


In [ ]:
# ---- ideal 1-D chain: exact piecewise-uniform kinematics ----------------
def T_of(y0, vy, mz, tau, s):
    """Flight time birth->detector: field-free lag tau, then E1|E2|drift|
    mirror(x s)|drift. y0 mm, vy mm/us; returns us (includes tau)."""
    m = mz*KG_AMU
    a1, a2 = (E_CHG*Ei/m*1e-6 for Ei in (E1, E2))     # V/mm -> mm/us^2
    am = E_CHG*EM0*s/m*1e-6
    y = y0 + vy*tau                                   # drift during the lag
    t1 = (-vy + np.sqrt(vy*vy + 2*a1*(yG1-y)))/a1
    v1 = vy + a1*t1
    v2 = np.sqrt(v1*v1 + 2*a2*(yG2-yG1))
    return (tau + t1 + (v2-v1)/a2 + (y_ent-yG2)/v2
            + 2*v2/am + (y_ent-y_det)/v2)

def fwhm_ns(mz, tau, s, Tk, box_y, n=4000, seed=7):
    rng = np.random.default_rng(seed)
    y0 = rng.uniform(10-box_y/2, 10+box_y/2, n)
    vy = rng.normal(0, math.sqrt(KB*Tk/(mz*KG_AMU))/1000.0, n)
    T = T_of(y0, vy, mz, tau, s)
    return 2.3548*T.std()*1e3, T.mean()-tau           # width; time FROM PULSE

# validation against the certified static flight (77 K, 0.5 mm sheet)
for mz, cert in ((100., 0.76), (500., 1.82)):
    f, _ = fwhm_ns(mz, 0.0, 1.0, 77.0, 0.5)
    print(f"m/z {mz:3.0f}: ideal {f:.2f} ns  vs certified flight {cert} ns")

In [ ]:
# ---- why tau is not a standalone knob: the regime map -------------------
print("m/z 500, best (tau, s) per regime | baseline -> tuned FWHM (ns)")
def autotune(mz, Tk, box_y, tau_hi=8.0):
    best = (1e9, (0.0, 1.0))
    for tau in np.linspace(0, tau_hi, 41):
        for s in np.linspace(0.95, 1.08, 53):
            f, _ = fwhm_ns(mz, tau, s, Tk, box_y)
            if f < best[0]: best = (f, (tau, s))
    (tau, s) = best[1]                                # polish
    for ta in np.arange(max(0, tau-0.15), tau+0.16, 0.01):
        for ss in np.arange(s-0.004, s+0.0042, 0.0002):
            f, _ = fwhm_ns(mz, ta, ss, Tk, box_y)
            if f < best[0]: best = (f, (ta, ss))
    return best[1], best[0]

for Tk, box in ((77, 0.5), (300, 0.5), (300, 0.1), (77, 0.1)):
    b, _ = fwhm_ns(500., 0, 1, Tk, box)
    (tau, s), f = autotune(500., Tk, box)
    print(f"  {Tk:3d} K, sheet {box} mm: {b:5.2f} -> {f:5.2f}  "
          f"(tau {tau:.2f} us, mirror x{s:.4f})")
print()
print("Read it off: at 0.5 mm the lag barely helps (and costs the off-mass);")
print("at 0.1 mm it is transformative, and 300 K/0.1 mm BEATS 77 K/0.5 mm --")
print("the lag substitutes for cooling. Thin sheet first, then lag.")

In [ ]:
# ---- the design point, and what one pulse costs the other mass ----------
TK, BOX = 77.0, 0.1
(tau, s), f5 = autotune(500., TK, BOX)
f1, Tp1 = fwhm_ns(100., tau, s, TK, BOX)
_, Tp5 = fwhm_ns(500., tau, s, TK, BOX)
print(f"design point: tau {tau:.2f} us, mirror x{s:.4f}  ({TK:.0f} K, {BOX} mm sheet)")
print(f"  m/z 500: {f5:.2f} ns  -> R {Tp5/(2*f5*1e-3):5.0f}   (design mass)")
print(f"  m/z 100: {f1:.2f} ns  -> R {Tp1/(2*f1*1e-3):5.0f}   (same pulse: the trade)")
print("Certified solver flight at (2.40, 1.0400): 0.83 ns / R 8473 and")
print("1.13 ns / R 2769 -- the ideal model under-predicts width by the")
print("solver-field residuals (fringe + rail ripple), stated not hidden.")

In [ ]:
# ---- build the pulsed spec from a (tau, mirror-scale) pair --------------
# One function builds ANY pulsed variant, so the tuned instrument and the
# untuned baseline below are guaranteed to differ only in (tau, s) -- the
# comparison cannot accidentally compare two different instruments.
import copy
from ion_gym.io.sim_spec import SimSpec
from ion_gym.physics.sim_build import build_run
from ion_gym.physics import ensemble_driver as ed

# ENSEMBLE SIZE. A resolving power is read off the WIDTH of an arrival
# distribution, so the ensemble has to be big enough to have a shape:
# at a couple of dozen ions the histogram is a few spikes and the peak
# it is supposed to show does not exist. Cost is not the constraint --
# measured 2 ms/ion on this deck, so both variants below are seconds.
N_IONS_PER_MASS = 600
# sigma -> FWHM for a Gaussian, named once and used everywhere below.
import math as _math
K_FWHM = 2.0 * _math.sqrt(2.0 * _math.log(2.0))     # = 2.3548...
# DECLARED seed: the deck is unseeded (fresh entropy per run), which is
# right for a user but wrong for a teaching figure whose numbers are
# quoted in the prose beneath it.
ENSEMBLE_SEED = 7

def build_pulsed_spec(tau_us, mirror_scale, n_ions=None, seed=None):
    """Pulsed oa deck at lag tau_us (us) and mirror scale mirror_scale.
    dt is PINNED at 1.0 ns because auto-dt misreads table drives."""
    nd = copy.deepcopy(spec_d)
    step = dict(frequency_hz=1.0, phase_deg=0.0, waveform="table",
                table_t_us=[0.0, round(tau_us, 3)], table_v=[0.0, 1.0],
                interp="hold", pe_mode="instant")
    nd["geometry"]["rf_groups"] = [dict(name="PUSH", amplitude_v=V_push, **step),
                                   dict(name="EXTR", amplitude_v=V_G1, **step)]
    for e in nd["geometry"]["electrodes"]:
        if e["name"] == "pusher":      e["rf_groups"] = ["PUSH"]; e["dc"] = 0.0
        if e["name"] == "extract_G1":  e["rf_groups"] = ["EXTR"]; e["dc"] = 0.0
        if e["name"].startswith("mir_ring") or e["name"] == "mirror_back":
            e["dc"] = round(e["dc"]*mirror_scale, 3)
        if e["name"] == "detector":    # lag shifts landing by v_x*tau per mass
            e["shapes"][0]["x_mm"] = 19.6; e["shapes"][0]["width_mm"] = 12.6  # trimmed frame (x' = x - 55)
    nd["source"]["box_mm"] = [3.0, BOX, 0.0]; nd["source"]["temperature_k"] = TK
    nd["source"]["n_ions"] = int(N_IONS_PER_MASS if n_ions is None else n_ions)
    nd["source"]["seed"] = int(ENSEMBLE_SEED if seed is None else seed)
    nd["integration"]["dt_ns"] = 1.0   # PIN dt: auto-dt misreads table drives
    sp = SimSpec.from_dict(nd)
    errs = sp.validate()
    if errs:
        raise ValueError("pulsed spec invalid: " + "; ".join(errs))
    return sp

def fly_pulsed(sp_variant, tau_us):
    """Fly a pulsed variant; return {mz: arrival-time array (us, lag
    subtracted)} for detected ions plus the ensemble object."""
    model, fly_fn, cols, births = build_run(sp_variant)
    prog = ed.run(births.shape[0], fly_fn, keep_full=False)
    tts = {}
    for mz in (100.0, 500.0):
        tts[mz] = np.array([m.summary["tof"]-tau_us for m in prog.results
                            if m.summary["mz"] == mz and m.summary["y_end"] > 30.5])
    return tts, prog

sp = build_pulsed_spec(tau, s)          # the TUNED instrument
print(f"quoted: pulsed-spec flight, {N_IONS_PER_MASS} ions/mass on a warm "
      f"field cache — a few seconds (measured 2 ms/ion); the next cell runs it.")


In [ ]:
# RUN CELL -- flies the TUNED pulsed spec quoted above (seconds, warm cache).
tts_tuned, prog = fly_pulsed(sp, tau)
for mz in (100.0, 500.0):
    tt = tts_tuned[mz]
    fw = 2.3548*tt.std()
    print(f"m/z {mz:3.0f}: {len(tt)}/{N_IONS_PER_MASS} | t {tt.mean():.4f} us  "
          f"FWHM {fw*1e3:.2f} ns  R {tt.mean()/(2*fw):5.0f}")
# Report what came from the deck and apply any override set above.
from ion_gym.io.deck_params import apply_deck_overrides
apply_deck_overrides(sp, **DECK_OVERRIDES)

# The spec here is built inside a helper, so the override hook does not
# apply -- but the reader still needs to see what the deck supplied.
from ion_gym.io.deck_params import describe_deck
describe_deck(sp)
print("quoted: the next cell REFLIES the untuned baseline (same cost) "
      "to draw the before/after comparison.")


## Resolving power: reading R off the distribution

Before comparing two tunes, be explicit about what the single number *R*
is and where it comes from, because a resolving power quoted without its
estimator is not checkable.

**The definition used everywhere in this notebook** is
**R = t̄ / (2·Δt_FWHM)** with **Δt_FWHM = 2√(2 ln 2)·σ ≈ 2.3548·σ**,
where t̄ and σ are the mean and standard deviation of the *detected*
arrival times, measured from the extraction pulse.

Two choices in that sentence are worth defending. First, the width comes
from **σ scaled to an equivalent FWHM**, not from reading half-maximum
crossings off the histogram: a literal crossing is a bin artifact — move
one ion and the crossing jumps a whole bin — while σ uses every sample.
Second, the figure below is **zoomed to ±5σ about the mean**, because at
full scale a sub-nanosecond peak sitting on a 6 or 14 µs arrival is
narrower than one pixel; the mean itself is printed in each panel so
nothing is hidden by the zoom.

The figure is the derivation, not an illustration of it: the histogram is
the flown ensemble, the blue curve is the Gaussian carrying that sample's
σ, the orange span is the FWHM that σ implies, and the box does the
division.

In [ ]:
# RUN CELL -- the anatomy of R, drawn on the TUNED ensemble already flown.
# Through the framework (viz_core.resolution_anatomy), not inline: the
# next notebook that needs to justify a resolving power gets this for
# free rather than rebuilding it.
from ion_gym.viz.viz_core import resolution_anatomy
from IPython.display import Image as _PNG, display as _display
import io as _io, matplotlib.pyplot as _plt

fig_R = resolution_anatomy(
    [dict(name="arrival time", unit="us", sample=tts_tuned[mz],
          delta_scale=1e3, delta_unit="ns",
          label=f"tuned, m/z {mz:.0f}") for mz in (100.0, 500.0)],
    title="How R is measured from the arrival-time distribution",
    operating_point=(f"tuned tau {tau:.2f} us, mirror x{s:.4f} | source "
                     f"{TK:.0f} K, {BOX} mm sheet | dt 1.0 ns pinned | "
                     f"{N_IONS_PER_MASS} ions/mass, seed {ENSEMBLE_SEED}"))
_buf = _io.BytesIO()
fig_R.savefig(_buf, format="png", dpi=110, bbox_inches="tight")
_display(_PNG(_buf.getvalue()))
_plt.close(fig_R)

for mz in (100.0, 500.0):
    tt = tts_tuned[mz]
    print(f"m/z {mz:3.0f}: n {tt.size:4d} | mean {tt.mean():.4f} us | "
          f"sigma {tt.std()*1e3:.3f} ns | FWHM {K_FWHM*tt.std()*1e3:.3f} ns "
          f"| R {tt.mean()/(2*K_FWHM*tt.std()):,.0f}")
print(f"statistical error on sigma at n={N_IONS_PER_MASS}: "
      f"~{100.0/ (2*(N_IONS_PER_MASS-1))**0.5:.1f}% -- R inherits it.")


## Before vs after: what the lag actually buys

The whole point of delayed extraction is below. **Before**: the pulse fires essentially immediately (lag = one integration step, mirror at the deck's own x1.0) — ions are extracted with their thermal velocity spread uncorrected, so two ions born at the same place but moving oppositely arrive at different times. **After**: the autotuned (τ, mirror) pair lets the initially-backward ions turn around and be caught by the field deeper in the well, which trades their velocity error against path length — the classic Wiley–McLaren cancellation. If the tuning did nothing, the two distributions below would lie on top of each other; the width ratio is the measured payoff, and the legend carries n, mean and spread for every sample.

In [ ]:
# RUN CELL -- baseline flight + before/after comparison figure.
# Baseline = prompt extraction: lag of ONE integration step (dt, the
# smallest representable lag -- derived, not invented) and the deck's
# own mirror (scale 1.0): the instrument before any time-lag tuning.
from ion_gym.viz.viz_core import distribution_compare
TAU_BASELINE_US = sp.integration.dt_ns * 1e-3      # = dt: prompt extraction
sp_base = build_pulsed_spec(TAU_BASELINE_US, 1.0)
tts_base, _ = fly_pulsed(sp_base, TAU_BASELINE_US)

def R_of(tt):
    """R = t / (2*FWHM), FWHM = 2sqrt(2 ln 2)*sigma -- ONE definition,
    used by every number and every figure in this notebook."""
    return tt.mean() / (2.0 * K_FWHM * tt.std())

panels = []
for mz in (100.0, 500.0):
    a, b = tts_base[mz], tts_tuned[mz]
    Ra, Rb = R_of(a), R_of(b)
    panels.append(dict(name=f"arrival time, m/z {mz:.0f}", unit="us",
                       # CENTRED: the two means differ by ~100 ns while the
                       # peaks are ~1 ns wide, so on a shared absolute axis
                       # each peak collapses into a single bin and the
                       # figure shows two bars instead of two lineshapes.
                       center="mean", delta_scale=1e3, delta_unit="ns",
                       bins=45,
                       samples={f"prompt, R {Ra:,.0f}": a,
                                f"tuned, R {Rb:,.0f} ({Rb/Ra:.1f}x)": b}))
    fa, fb = K_FWHM*a.std()*1e3, K_FWHM*b.std()*1e3
    print(f"m/z {mz:3.0f}: FWHM {fa:7.2f} -> {fb:7.2f} ns  "
          f"({fa/fb:4.1f}x narrower) | n {len(a)}/{len(b)} of "
          f"{N_IONS_PER_MASS} detected | R {Ra:5.0f} -> {Rb:5.0f}")
fig = distribution_compare(
    panels,
    title="Resolving power, before vs after autotune",
    operating_point=(f"prompt (tau {TAU_BASELINE_US*1e3:.0f} ns, mirror x1) vs "
                     f"tuned (tau {tau:.2f} us, mirror x{s:.4f}) | source "
                     f"{TK:.0f} K, {BOX} mm sheet | dt 1.0 ns pinned | "
                     f"{N_IONS_PER_MASS} ions/mass, seed {ENSEMBLE_SEED}"))
# Render deterministically: a bare trailing `fig` relies on the
# inline-backend display hook, and on a kernel without it the cell
# printed the repr string instead of the figure (
# measured). Explicit PNG bytes render on every backend.
from IPython.display import Image as _PNG, display as _display
import io as _io, matplotlib.pyplot as _plt
_buf = _io.BytesIO()
fig.savefig(_buf, format="png", dpi=110, bbox_inches="tight")
_display(_PNG(_buf.getvalue()))
_plt.close(fig)


### The same tune pair, seen as flights

Histograms compress each flight to one number; the paths show *why* the
numbers move. Both panels below are the **same instrument built by the
same function** — they differ only in (τ, mirror scale) — rendered from
the solver's own mask with a few example ions per mass, exactly as
flown. Before: prompt extraction fans the thermal sheet into a wide
arrival spread. After: the lag lets the sheet drift so the push field
imprints a position-dependent kick that time-focuses at the detector.

In [ ]:
# Before/after example flights, through the framework (viz_core): the
# SAME specs the histograms came from, reflown keeping trajectories for
# a handful of example ions per mass.
# Enough paths to show the BUNDLE, not a token few: the point of these
# two panels is how wide the packet is when it reaches the detector, and
# three lines cannot show a width.
N_EXAMPLE_PER_MASS = 12

def example_flights(sp_variant):
    """Build the variant and fly the first N_EXAMPLE_PER_MASS ions of
    each contiguous mass block, keeping trajectories."""
    model_v, fly_v, cols_v, births_v = build_run(sp_variant)
    n_per = births_v.shape[0] // len(sp_variant.source.mz_list)
    trajs, fates = [], []
    for blk in range(len(sp_variant.source.mz_list)):
        for j in range(N_EXAMPLE_PER_MASS):
            tr, s = fly_v(blk * n_per + j)
            if tr is not None and len(tr):
                trajs.append(tr)
                fates.append(str(s.get("kind", "")))
    return model_v, trajs, fates

from ion_gym.viz.viz_core import scene_from_simspec, render_mpl
from IPython.display import Image as _PNG, display as _display
import io as _io, matplotlib.pyplot as _plt

for label, sp_v, tau_v in ((f"BEFORE — prompt (tau = {TAU_BASELINE_US*1e3:.0f} ns, mirror x1)",
                            sp_base, TAU_BASELINE_US),
                           (f"AFTER — tuned (tau = {tau:.2f} us, mirror x{s:.4f})",
                            sp, tau)):
    model_v, trajs_v, fates_v = example_flights(sp_v)
    scene_v = scene_from_simspec(
        sp_v, model_v, field="phi", trajs=trajs_v, fates=fates_v,
        title=f"{label} | {2*N_EXAMPLE_PER_MASS} example ions "
              f"(m/z 100 + 500) | dt {sp_v.integration.dt_ns:g} ns")
    fig_v = render_mpl(scene_v, views=["xy"])
    _buf = _io.BytesIO()
    fig_v.savefig(_buf, format="png", dpi=100, bbox_inches="tight")
    _display(_PNG(_buf.getvalue(), height=740))
    _plt.close(fig_v)


### Why this R and a certified Rp are not the same number

The estimator is defined and drawn two sections above; what remains is
the caveat. At n = 600 per mass the relative statistical error on σ is
≈ 1/√(2(n−1)) ≈ **2.9%**, and R inherits it — so the before→after
*ratio*, which shares the ensemble, is the robust claim, not the third
digit of either R.

The certified resolving powers elsewhere in this repo use a different and
heavier convention — median arrival, a robust-percentile FWHM, an
explicit timing floor, and multiple seeds — so a notebook R and a
certified Rp must never be quoted interchangeably: same physics,
different estimator. Two further honesty notes on the numbers above. The
ideal 1-D model under-predicts the width, because it has no solver-field
residuals (fringe fields, rail ripple). And the m/z 100 R gets *worse*
under the tune — that is not a defect, it is finding (3) below: one pulse
is exact at one mass, and the trade has to be quoted, not hidden.

**Findings.** (1) τ and the mirror scale are one pair — the optimizer never
returns τ > 0 with s = 1, because the matched mirror gives converted position
nothing to act on. (2) The lag's value is set by the sheet thickness, not the
temperature: thin the packet before pulsing. (3) One pulse serves every mass;
the correction is exact at one — quote both masses or the number is theater.
(4) TOF is measured **from the pulse**; the lag itself buys no resolving
power. Next lever when this is exhausted: the two-segment (Mamyrin) ladder —
the side-rail mirror accepts it as a voltage-rule change only.

## Read-out

Delayed extraction trades one spread against another: waiting lets the ion cloud spatially sort by velocity, so the pulse that follows can time-focus a *correlated* spread instead of a random one. The optimum delay is therefore an ion-mass-dependent compromise, and a delay tuned for one m/z is detuned for another — which is why the scan, not a single number, is the deliverable.